# Notebook kerja Syaikha A (dataset dan visualize)
Sel berlabel **SALINAN SEMENTARA** cuma dipakai buat uji di notebook ini dan dibuang waktu ketua merakit `MiniDDPM.ipynb`. Bagian Syaikha A mulai dari "4. Dataset".

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
import math
import random
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

In [ ]:
# Import Teman A (bagian 1). Saat merakit, ketua menyatukannya ke sel import utama.
import copy
import tempfile

from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Root repo: folder kerja saat ini, atau folder induknya kalau dijalankan dari notebooks/
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
config = SimpleNamespace(
    # Reproducibility
    SEED=42,  # dipakai set_seed(); satu seed untuk semua anggota
    # Training (baseline)
    LEARNING_RATE=2e-4,  # baseline; boleh ditimpa konfigurasi di EXPERIMENTS
    # SEMENTARA, BELUM FINAL: menunggu uji memori di GTX 1650 Ti (4GB).
    # Jangan dipakai untuk eksperimen tuning sebelum nilai final ditetapkan.
    BATCH_SIZE=64,
    EPOCHS=30,
    NUM_WORKERS=0,  # DataLoader tanpa proses worker (aman di macOS dan Windows)
    # Diffusion (baseline)
    T=1000,  # jumlah timestep baseline; boleh ditimpa konfigurasi di EXPERIMENTS
    # Acuan untuk T=1000; Diffusion menskalakan beta dengan BETA_REFERENCE_T / T
    BETA_START=1e-4,
    BETA_END=0.02,
    BETA_REFERENCE_T=1000,
    # Model
    IMAGE_SIZE=28,  # sisi gambar MNIST; channel ada di MODEL_PARAMS
    # Dipakai sebagai UNet(**config.MODEL_PARAMS) dan disimpan di checkpoint.
    # channel_mults (1, 2, 2) = dua downsampling: 28 -> 14 -> 7
    MODEL_PARAMS={
        "in_channels": 1,
        "base_channels": 64,
        "channel_mults": (1, 2, 2),
        "time_emb_dim": 128,
    },
)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Sampling selama training
config.SAMPLE_EVERY_N_EPOCHS = 5  # grid sampel disimpan tiap N epoch
config.NUM_SAMPLE_IMAGES = 16     # jumlah gambar per grid

# Smoke test (uji alur cepat, bukan untuk hasil)
config.SMOKE_TEST = False         # True = subset kecil dan epoch pendek
config.SMOKE_SUBSET_SIZE = 512    # jumlah gambar pada mode smoke test
config.SMOKE_EPOCHS = 2           # jumlah epoch pada mode smoke test
config.SMOKE_SAMPLE_EVERY = 1     # interval grid smoke test (harus <= SMOKE_EPOCHS)

# Path: semuanya diturunkan dari BASE_DIR (sel sebelumnya)
config.BASE_DIR = BASE_DIR
config.DATA_DIR = BASE_DIR / "data"               # unduhan MNIST (tidak di-commit)
config.CHECKPOINT_DIR = BASE_DIR / "checkpoints"  # checkpoint (tidak di-commit)
config.RESULTS_DIR = BASE_DIR / "results"         # hasil (di-commit)
config.LOSS_CURVES_DIR = config.RESULTS_DIR / "loss_curves"
config.SAMPLES_DIR = config.RESULTS_DIR / "samples"

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
def set_seed(seed):
    """Seed random, numpy, torch. Tanpa mode deterministik penuh agar training tidak melambat."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # sekaligus menyemai CUDA dan MPS

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
def get_device():
    """Device pertama yang tersedia: cuda, mps, cpu. Tanpa print."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

## 4. Dataset (Syaikha A)
MNIST diunduh otomatis ke `data/`, lalu tiap gambar diubah ke rentang [-1, 1] supaya sama dengan skala noise Gaussian yang dipakai diffusion. Model kita unconditional (nggak dikasih tahu digit berapa), jadi labelnya dibuang di sini dan tiap batch cuma berisi gambar.

In [ ]:
class ImagesOnly(Dataset):
    """Bungkus dataset (gambar, label) supaya hanya mengembalikan gambar."""

    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, _label = self.base[idx]  # label dibuang di sini
        return image

`get_dataloader` membaca semua pengaturan dari argumen `cfg`, bukan dari `config` global. Jadi `train` milik Teman B bisa ngirim salinan config dengan `SMOKE_TEST` sendiri. Kalau mode smoke nyala, yang dipakai cuma `SMOKE_SUBSET_SIZE` gambar pertama biar ujinya cepat.

In [ ]:
def get_dataloader(cfg):
    """DataLoader MNIST: tiap batch satu tensor float (B, 1, 28, 28) dalam [-1, 1], tanpa label.
    Semua nilai dibaca dari cfg; data tidak dipindah ke device (itu tugas train)."""
    to_range = transforms.Compose([
        transforms.ToTensor(),                 # piksel 0..255 -> float [0, 1], shape (1, 28, 28)
        transforms.Normalize((0.5,), (0.5,)),  # (x - 0.5) / 0.5 -> [-1, 1]
    ])
    mnist = datasets.MNIST(root=cfg.DATA_DIR, train=True, download=True, transform=to_range)
    data = ImagesOnly(mnist)
    if cfg.SMOKE_TEST:
        data = Subset(data, range(min(cfg.SMOKE_SUBSET_SIZE, len(data))))
    return DataLoader(
        data,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        drop_last=False,
    )

Sel uji ini gantinya `python -m scripts.check_data`: ngecek shape, tipe, dan rentang satu batch, mastiin nggak ada label yang ikut, lalu nampilin beberapa contoh digit.

In [ ]:
# Uji dataset (mode penuh): shape, tipe, rentang, tanpa label
snapshot = copy.deepcopy(vars(config))
cfg_full = SimpleNamespace(**copy.deepcopy(vars(config)))
cfg_full.SMOKE_TEST = False
loader = get_dataloader(cfg_full)
batch = next(iter(loader))

assert isinstance(batch, torch.Tensor), "batch harus satu tensor, bukan (gambar, label)"
expected = (cfg_full.BATCH_SIZE, config.MODEL_PARAMS["in_channels"], config.IMAGE_SIZE, config.IMAGE_SIZE)
assert tuple(batch.shape) == expected, f"shape salah: {tuple(batch.shape)}"
assert batch.dtype == torch.float32
assert batch.min() >= -1.0 and batch.max() <= 1.0, "nilai harus dalam [-1, 1]"
assert batch.device.type == "cpu", "dataset tidak boleh memindahkan data ke device"
assert len(loader.dataset) == 60000, "MNIST train harus 60000 gambar"

print(f"jumlah gambar : {len(loader.dataset)}  |  jumlah batch: {len(loader)}")
print(f"shape batch   : {tuple(batch.shape)}  dtype: {batch.dtype}")
print(f"rentang nilai : min={batch.min().item():.3f}  max={batch.max().item():.3f}")

In [ ]:
# Uji mode smoke: subset kecil, shape tetap sama, config asli tidak berubah
cfg_smoke = SimpleNamespace(**copy.deepcopy(vars(config)))
cfg_smoke.SMOKE_TEST = True
smoke_loader = get_dataloader(cfg_smoke)
smoke_batch = next(iter(smoke_loader))

assert len(smoke_loader.dataset) == config.SMOKE_SUBSET_SIZE, "subset smoke harus SMOKE_SUBSET_SIZE gambar"
assert smoke_batch.shape == batch.shape, "shape batch smoke harus sama dengan mode penuh"
assert smoke_batch.min() >= -1.0 and smoke_batch.max() <= 1.0
n_seen = sum(b.shape[0] for b in smoke_loader)
assert n_seen == config.SMOKE_SUBSET_SIZE, "satu epoch smoke harus melewati semua gambar subset"
assert vars(config) == snapshot, "config tidak boleh berubah"
print(f"smoke: {len(smoke_loader.dataset)} gambar, {len(smoke_loader)} batch, shape {tuple(smoke_batch.shape)}")

In [ ]:
# Contoh gambar dari satu batch (dikembalikan ke [0, 1] hanya untuk ditampilkan)
fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img in zip(axes, batch[:8]):
    ax.imshow(((img[0] + 1) / 2).numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
plt.show()
print("dataset OK")

## 5. Visualize (Teman A)
Dua fungsi buat nyimpen hasil: `save_image_grid` untuk grid gambar dan `plot_loss` untuk grafik loss. Keduanya nyimpen ke file sekaligus nampilin di notebook, dan dipakai `train` serta `sample` milik Teman B.

Gambar dari model ada di rentang [-1, 1], sedangkan file PNG butuh [0, 1], jadi dikonversi dulu dengan `(x + 1) / 2`. Tensor juga dipindah ke CPU dulu, karena hasil sampling bisa masih ada di GPU.

In [ ]:
def save_image_grid(images, path):
    """Simpan tensor (N, 1, 28, 28) dalam [-1, 1] sebagai satu grid PNG, lalu tampilkan di notebook."""
    images = images.detach().cpu().float().clamp(-1.0, 1.0)
    images = (images + 1.0) / 2.0  # [-1, 1] -> [0, 1]
    nrow = math.ceil(math.sqrt(images.shape[0]))  # grid sepersegi mungkin
    grid = make_grid(images, nrow=nrow, padding=2, pad_value=1.0)  # (3, H, W)
    grid_np = grid.permute(1, 2, 0).numpy()

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.imsave(path, grid_np)

    fig, ax = plt.subplots(figsize=(max(2, nrow * 0.9), max(2, nrow * 0.9)))
    ax.imshow(grid_np)
    ax.set_title(path.name, fontsize=8)
    ax.axis("off")
    plt.show()
    plt.close(fig)

`plot_loss` menerima dict `{label: [loss per epoch]}`, jadi bisa satu garis (satu konfigurasi) atau beberapa garis sekaligus buat bandingin eksperimen. Kalau dikasih list biasa, otomatis dianggap satu seri bernama "loss".

In [ ]:
def plot_loss(series, path):
    """Grafik loss per epoch dari {label: [loss, ...]} (atau satu list), simpan ke file lalu tampilkan."""
    if not isinstance(series, dict):
        series = {"loss": list(series)}
    fig, ax = plt.subplots(figsize=(6, 4))
    for label, losses in series.items():
        epochs = range(1, len(losses) + 1)
        ax.plot(epochs, losses, marker="o", markersize=3, label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE loss")
    ax.set_title("Loss training per epoch")
    ax.grid(alpha=0.3)
    ax.legend()

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.show()
    plt.close(fig)

In [ ]:
# Uji save_image_grid: tensor acak, gambar MNIST asli, dan tensor yang masih butuh gradien
with tempfile.TemporaryDirectory() as tmp:
    rand_path = Path(tmp) / "sub" / "grid_acak.png"  # folder belum ada -> harus dibuat
    save_image_grid(torch.rand(16, 1, 28, 28) * 2 - 1, rand_path)
    assert rand_path.exists() and rand_path.stat().st_size > 0

    mnist_path = Path(tmp) / "grid_mnist.png"
    save_image_grid(batch[:16], mnist_path)
    saved = plt.imread(mnist_path)
    assert saved.min() >= 0.0 and saved.max() <= 1.0, "gambar tersimpan harus dalam [0, 1]"

    grad_path = Path(tmp) / "grid_grad.png"
    save_image_grid(torch.zeros(4, 1, 28, 28, requires_grad=True), grad_path)  # tidak boleh error
    assert grad_path.exists()
print("save_image_grid OK")

In [ ]:
# Uji plot_loss: satu seri, beberapa seri (perbandingan eksperimen), dan list biasa
with tempfile.TemporaryDirectory() as tmp:
    one = Path(tmp) / "loss_satu.png"
    plot_loss({"baseline": [0.9, 0.5, 0.3, 0.2, 0.15]}, one)
    assert one.exists() and one.stat().st_size > 0

    many = Path(tmp) / "loss_banyak.png"
    plot_loss({"baseline": [0.9, 0.5, 0.3], "lr_1e-4": [1.0, 0.7, 0.5], "lr_5e-4": [0.8, 0.4, 0.35]}, many)
    assert many.exists()

    plain = Path(tmp) / "loss_list.png"
    plot_loss([0.6, 0.4, 0.3], plain)
    assert plain.exists()
print("plot_loss OK")
print("visualize OK")